# How steep is the 200 GeV jet spectrum, and what does that do to R_AA?

**The question** (2026-11-25, phase 3 of the O+O physics run). The model's quenching factor for
sPHENIX jets looked as if it fell at high jet p_T, the opposite of the A+A intuition. One explanation
is kinematic. For a local power law dσ/dp_T ∝ p_T^(−n), losing a fixed fraction ε of the jet energy
gives a suppression (1 − ε)^(n−1). At √s = 200 GeV the jets approach the kinematic limit
(x_T = 2p_T/√s goes from 0.15 to 0.7 over 15–70 GeV), so n should grow with p_T. Is that large in
the measured p+p jet cross sections, and does our p+p reference have it?

**Inputs**
- **sPHENIX p+p, preliminary** (sPH-CONF-JET-2025-03): anti-k_T R = 0.4, |η| < 0.7, unfolded to
  particle level, 15–72 GeV. Read from the note's vector Fig. 5 (`refs/sphenix_pp_jet`). The drawn
  statistical bars are the same in every bin, so they are upper bounds; the systematics are largely
  correlated (JES).
- **STAR p+p** (arXiv:2603.28695, HEPData ins3137518): anti-k_T R = 0.5, |η| < 0.5 and 0.5 < |η| < 0.9,
  6.9–52 GeV. These are the hard tune's jet data.
- **Our p+p reference**: `pp_ref`, all finished jobs (phase 3 + the 3e windows at 35–40 and 47–55 GeV; the full chain, `pp200_isr_rhictune.xml`, 70k events). The O+O arms (sections 3–5) use all jobs too: phase 3 + 3b + 3e, 222.5k events each.
  Full jets R = 0.4, |η| < 0.7, as for sPHENIX, stitched over the p̂_T windows, absolute
  (σ_k/N_k from Pythia, mb → pb).

**Method**
- **Fit:** σ(p_T) = A p_T^(−n₀) (1 − x_T)^m, integrated over each bin (the bins are wide, and at
  n ≈ 14 the bin average differs from the value at the centre by up to 20 %). Fit weights are
  stat ⊕ syst, so the fit describes the shape; its χ² is not a test.
- **Local power from the fit:** n(p_T) = −d ln σ/d ln p_T = n₀ + m x_T/(1 − x_T). The (1 − x_T)^m
  factor is the falling parton densities at large x.
- **Local power, point by point:** n between neighbouring bins, at the Lafferty–Wyatt positions from
  the fit, with statistical errors only, since the systematics move neighbours together.

Model results, not vetted. The data are partly preliminary.

In [ ]:
import sys as _sys
_sys.path.insert(0, "..")
import _paths  # noqa: E402,F401  (analysis/OO-RHIC and its test directories on sys.path; shared results/, refs/, data/ in ..)
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from scipy.integrate import quad

SQRTS = 200.0
HERE = Path(".")
# --- data
sp = json.loads((HERE / "../refs/sphenix_pp_jet/sphenix_pp_jet_prelim.json").read_text())["rows"]
D = {}
D["sPHENIX R=0.4 |η|<0.7 (prel.)"] = dict(
    lo=np.array([r["lo"] for r in sp]), hi=np.array([r["hi"] for r in sp]), y=np.array([r["value"] for r in sp]),
    stat=np.array([(r["stat_lo"] + r["stat_hi"]) / 2 for r in sp]), sys=np.array([(r["sys_lo"] + r["sys_hi"]) / 2 for r in sp]),
    err_lo=np.array([np.hypot(r["stat_lo"], r["sys_lo"]) for r in sp]), err_hi=np.array([np.hypot(r["stat_hi"], r["sys_hi"]) for r in sp]))
dd = json.loads((HERE / "../data/data.json").read_text())
for k, lab in (("star_jets_eta0", "STAR R=0.5 |η|<0.5"), ("star_jets_eta1", "STAR R=0.5 0.5<|η|<0.9")):
    t = dd[k]; e = np.array(t["edges"])
    D[lab] = dict(lo=e[:, 0], hi=e[:, 1], y=np.array(t["value"]), stat=np.array(t["stat"]), sys=np.array(t["sys"]))
# --- model: pp_ref full jets R = 0.4, |eta| < 0.7 (all jobs), absolute
RES = "../results/oo_analysis_rhic.json"                             # oo_analysis_rhic.py (all jobs)
ev = dict(np.load("../results/oo_analysis_rhic_pp_ref.npz"))
E = np.array(json.loads(Path(RES).read_text())["edges"]["sph_jet"])
S = (ev["w"][:, None] * ev["sph_jet"]).sum(0); S2 = ((ev["w"][:, None] * ev["sph_jet"]) ** 2).sum(0)
conv = 1e9 / np.diff(E) / (2 * 0.7)                                    # mb -> pb, per GeV, per unit eta
D["model pp_ref R=0.4 |η|<0.7"] = dict(lo=E[:-1], hi=E[1:], y=S * conv, stat=np.sqrt(S2) * conv, sys=np.zeros(len(S)))
print("model pp_ref:", len(ev["w"]), "events;", "windows", [f"{a:g}-{b:g}" for a, b in ev["windows"]])
for k, d in D.items():
    print(f"{k:32s} {len(d['y'])} bins, {d['lo'][0]:g}-{d['hi'][-1]:g} GeV")

In [ ]:
def f(pt, A, n0, m):
    xt = 2 * pt / SQRTS
    return np.exp(A) * pt ** (-n0) * np.clip(1 - xt, 1e-9, None) ** m

def binavg(p, lo, hi):
    return np.array([quad(lambda x: f(x, *p), a, b)[0] / (b - a) for a, b in zip(lo, hi)])

def fit(d):
    err = np.hypot(d["stat"], d["sys"]); err = np.where(err > 0, err, 0.05 * d["y"])
    r = lambda p: (np.log(binavg(p, d["lo"], d["hi"])) - np.log(d["y"])) / (err / d["y"])
    p0 = [np.log(d["y"][0]) + 5 * np.log(d["lo"][0]), 5.0, 5.0]
    res = least_squares(r, p0)
    J = res.jac; cov = np.linalg.pinv(J.T @ J) * max(1.0, 2 * res.cost / max(len(d["y"]) - 3, 1))
    return res.x, cov, 2 * res.cost, len(d["y"]) - 3

def nloc(p, pt):
    xt = 2 * pt / SQRTS
    return p[1] + p[2] * xt / (1 - xt)

def lafferty_wyatt(p, lo, hi):
    out = []
    for a, b in zip(lo, hi):
        avg = quad(lambda x: f(x, *p), a, b)[0] / (b - a)
        xs = np.linspace(a, b, 2001); out.append(xs[np.argmin(np.abs(f(xs, *p) - avg))])
    return np.array(out)

def fit_simple(lo, hi, y, e):
    return fit(dict(lo=lo, hi=hi, y=y, stat=e, sys=np.zeros_like(y)))[0]

FITS, PTS = {}, {}
for k, d in D.items():
    p, cov, chi2, ndf = fit(d)
    FITS[k] = (p, cov)
    xl = lafferty_wyatt(p, d["lo"], d["hi"])
    n_pt = -np.diff(np.log(d["y"])) / np.diff(np.log(xl))
    en = np.sqrt((d["stat"][1:] / d["y"][1:]) ** 2 + (d["stat"][:-1] / d["y"][:-1]) ** 2) / np.diff(np.log(xl))
    PTS[k] = (np.sqrt(xl[1:] * xl[:-1]), n_pt, en)
    print(f"{k:32s} n0 = {p[1]:5.2f} ± {np.sqrt(cov[1,1]):.2f}, m = {p[2]:5.2f} ± {np.sqrt(cov[2,2]):.2f}  "
          f"(chi2 {chi2:.1f} / {ndf}, stat+syst weights)")
print()
grid = np.array([10, 15, 20, 25, 30, 40, 50, 60, 70.0])
print("local power n(pT) from the fits:")
print("  pT [GeV]                        " + " ".join(f"{x:5.0f}" for x in grid))
for k, (p, cov) in FITS.items():
    lo, hi = D[k]["lo"][0], D[k]["hi"][-1]
    print(f"  {k:31s} " + " ".join(f"{nloc(p, x):5.1f}" if lo - 2 <= x <= hi + 2 else "    -" for x in grid))

In [ ]:
COL = {"sPHENIX R=0.4 |η|<0.7 (prel.)": "k", "STAR R=0.5 |η|<0.5": "C0", "STAR R=0.5 0.5<|η|<0.9": "C9",
       "model pp_ref R=0.4 |η|<0.7": "C3"}
fig, ax = plt.subplots(1, 3, figsize=(17, 4.8))
for k, d in D.items():
    x = 0.5 * (d["lo"] + d["hi"]); p, _ = FITS[k]
    yerr = [d["err_lo"], d["err_hi"]] if "err_lo" in d else np.hypot(d["stat"], d["sys"])
    ax[0].errorbar(x, d["y"], yerr, 0.5 * (d["hi"] - d["lo"]), fmt="o", ms=4, color=COL[k], label=k)
    xx = np.linspace(d["lo"][0], d["hi"][-1], 200); ax[0].plot(xx, f(xx, *p), color=COL[k], lw=0.8)
ax[0].set_yscale("log"); ax[0].set_xlabel("jet $p_T$ [GeV]"); ax[0].set_ylabel(r"$d^2\sigma/dp_T d\eta$ [pb/GeV]")
ax[0].legend(fontsize=7); ax[0].set_title("p+p 200 GeV jet cross sections (lines: fits)", fontsize=9)
# model / data in each experiment's own jet definition and bins (pp_jet_spectra.py: pp_ref, all jobs);
# points: model in the same bins / data (bars: model statistics; shaded: the data's total uncertainty);
# lines: model fit / data fit
ks, km = "sPHENIX R=0.4 |η|<0.7 (prel.)", "model pp_ref R=0.4 |η|<0.7"
MS = json.loads(Path("../results/pp_jet_spectra.json").read_text())
PAIRS = (("sphenix", "sPHENIX R=0.4 |η|<0.7 (prel.)", "k"), ("star_eta0", "STAR R=0.5 |η|<0.5", "C0"),
         ("star_eta1", "STAR R=0.5 0.5<|η|<0.9", "C9"))
RATIO = {}
for mk_, dk_, c in PAIRS:
    d = D[dk_]; mm = MS[mk_]; x = 0.5 * (d["lo"] + d["hi"])
    r = np.array(mm["value"]) / d["y"]; er = np.array(mm["err"]) / d["y"]
    lo_, hi_ = (d["err_lo"], d["err_hi"]) if "err_lo" in d else (np.hypot(d["stat"], d["sys"]),) * 2
    for xl, xh, a, b in zip(d["lo"], d["hi"], 1 - lo_ / d["y"], 1 + hi_ / d["y"]):
        ax[1].fill_between([xl, xh], a, b, color=c, alpha=0.08, lw=0)
    ax[1].errorbar(x, r, er, 0.5 * (d["hi"] - d["lo"]), fmt="o", ms=4, color=c, label=f"model / {dk_}")
    xx = np.linspace(d["lo"][0], d["hi"][-1], 200)
    pmod = fit_simple(np.array(mm["edges"][:-1]), np.array(mm["edges"][1:]), np.array(mm["value"]), np.array(mm["err"]))
    ax[1].plot(xx, f(xx, *pmod) / f(xx, *FITS[dk_][0]), color=c, lw=0.9)
    RATIO[dk_] = (d["lo"], d["hi"], r, er)
ax[1].axhline(1, color="k", lw=0.8); ax[1].set_ylim(0, 2.2); ax[1].set_xlabel("jet $p_T$ [GeV]"); ax[1].set_ylabel("model / data")
ax[1].legend(fontsize=7); ax[1].set_title("model p+p (same jet definition and bins) / data; lines: fit / fit", fontsize=9)
for k, (p, cov) in FITS.items():
    xx = np.linspace(max(D[k]["lo"][0], 8), D[k]["hi"][-1], 200)
    ax[2].plot(xx, nloc(p, xx), color=COL[k], label=k)
    xp, n_pt, en = PTS[k]
    ax[2].errorbar(xp, n_pt, en, fmt="o", ms=3, color=COL[k], alpha=0.7)
ax[2].set_xlabel("jet $p_T$ [GeV]"); ax[2].set_ylabel(r"local power $n = -d\ln\sigma/d\ln p_T$"); ax[2].set_ylim(2, 22)
ax[2].legend(fontsize=7); ax[2].set_title("lines: fits n₀ + m x_T/(1−x_T); points: neighbouring bins (stat)", fontsize=9)
fig.tight_layout(); plt.show()
for k_, (lo_, hi_, r_, er_) in RATIO.items():
    print(f"model / {k_}: " + " ".join(f"{a:g}-{b:g}:{v:.2f}±{e:.2f}" for a, b, v, e in zip(lo_, hi_, r_, er_)))

### STAR against sPHENIX

The two measure different jets: STAR uses R = 0.5 at |η| < 0.5 (or 0.5–0.9), sPHENIX R = 0.4 at
|η| < 0.7. The left panel shows the data ratio, STAR's points over sPHENIX's fit integrated over
STAR's bins, with STAR's total uncertainty. The lines are the same ratio in the model, using
`pp_jet_spectra.py`'s jets in both definitions: what the definitions alone make of it. The right
panel divides the two, giving the disagreement between the experiments with the definitions taken
out by the model. sPHENIX's own systematics (correlated, +50 / −26 % in the lowest bin, about
±60–100 % at high p_T) are not in the bars. The shaded band is their size at STAR's bin centres.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
dS = D[ks]; pSf = FITS[ks][0]
# sPHENIX's relative total uncertainty, interpolated to a pT (log-linear between its bin centres)
xs_ = 0.5 * (dS["lo"] + dS["hi"])
rel_lo = np.interp(np.log(1), [0], [0])  # placeholder to keep names explicit
def s_rel(x, side):
    r_ = (dS["err_lo"] if side == "lo" else dS["err_hi"]) / dS["y"]
    return np.interp(x, xs_, r_)
msp = MS["sphenix"]
pm_sp = fit_simple(np.array(msp["edges"][:-1]), np.array(msp["edges"][1:]), np.array(msp["value"]), np.array(msp["err"]))
for mk_, dk_, c, dx in (("star_eta0", "STAR R=0.5 |η|<0.5", "C0", -0.3), ("star_eta1", "STAR R=0.5 0.5<|η|<0.9", "C9", 0.3)):
    d = D[dk_]; x = 0.5 * (d["lo"] + d["hi"]); ok = (d["lo"] >= dS["lo"][0] - 1e-6)
    sp_b = binavg(pSf, d["lo"], d["hi"])
    r = d["y"] / sp_b; er = r * np.hypot(d["stat"], d["sys"]) / d["y"]
    mm = MS[mk_]; mod_b = np.array(mm["value"]) / binavg(pm_sp, d["lo"], d["hi"])
    emod = np.array(mm["err"]) / binavg(pm_sp, d["lo"], d["hi"])
    ax[0].errorbar(x[ok] + dx, r[ok], er[ok], 0.5 * (d["hi"] - d["lo"])[ok], fmt="o", ms=4, color=c, label=f"{dk_} / sPHENIX (data)")
    ax[0].plot(x[ok], mod_b[ok], "-", color=c, lw=1.2, label=f"model, same definitions ({dk_[11:]})")
    dr = r / mod_b; edr = dr * np.hypot(er / r, emod / mod_b)
    ax[1].errorbar(x[ok] + dx, dr[ok], edr[ok], 0.5 * (d["hi"] - d["lo"])[ok], fmt="o", ms=4, color=c, label=dk_)
    print(f"{dk_:24s} data ratio:  " + " ".join(f"{a:g}-{b:g}:{v:.2f}" for a, b, v, o in zip(d["lo"], d["hi"], r, ok) if o))
    print(f"{'':24s} model ratio: " + " ".join(f"{a:g}-{b:g}:{v:.2f}" for a, b, v, o in zip(d["lo"], d["hi"], mod_b, ok) if o))
    print(f"{'':24s} data/model:  " + " ".join(f"{a:g}-{b:g}:{v:.2f}±{e:.2f}" for a, b, v, e, o in zip(d["lo"], d["hi"], dr, edr, ok) if o))
xx = np.linspace(15, 52, 100)
ax[1].fill_between(xx, 1 - s_rel(xx, "lo"), 1 + s_rel(xx, "hi"), color="k", alpha=0.08, lw=0, label="sPHENIX total uncertainty")
for a_ in ax:
    a_.axhline(1, color="k", lw=0.8); a_.set_xlabel("jet $p_T$ [GeV]"); a_.legend(fontsize=7); a_.set_xlim(14, 54)
ax[0].set_ylim(0, 2.5); ax[1].set_ylim(0, 2.5)
ax[0].set_ylabel("STAR / sPHENIX"); ax[1].set_ylabel("(STAR / sPHENIX)_data / (same)_model")
ax[0].set_title("data ratio (points) and what the jet definitions alone give (model, lines)", fontsize=9)
ax[1].set_title("experiments' disagreement with the definitions removed", fontsize=9)
fig.tight_layout(); plt.show()

## What the steepening does to the suppression

If the medium removes a fixed fraction ε of every jet's p_T, the yield at p_T comes from p+p jets at
p_T/(1−ε), and the suppression is Q(p_T) ≈ (1 − ε)^(n(p_T)−1). The same holds for a constant
absolute loss ΔE, with ε = ΔE/p_T falling with p_T. The left panel shows Q against p_T for
fixed ε with sPHENIX's measured n(p_T). The right panel turns quenching factors back into ε:
- for the data, sPHENIX R_CP (0–20/60–80 %) divided by the selection bias (Angantyr from the O+O
  note, and ours, `oo_vac`);
- for the model, `oo_med`/`oo_vac` in 0–20 % by impact parameter and R_CP's double ratio.

An R_CP is quenching relative to the 60–80 % class (itself slightly quenched), so the ε are
central-minus-peripheral losses. Read them as an effective fractional loss, not a measured ΔE.

In [ ]:
pS, _ = FITS[ks]
R = json.loads(Path(RES).read_text())
SPD = R["data"]["sphenix"]; TH = SPD.get("theory", {})
rows = SPD["rcp_vs_60_80"]["red"]
xd = np.array([(r_["lo"] + r_["hi"]) / 2 for r_ in rows]); vd = np.array([r_["value"] for r_ in rows])
ed = np.array([(r_["stat_lo"] + r_["stat_hi"]) / 2 for r_ in rows])
ang = TH.get("angantyr_bias_rcp_vs_60_80", {}).get("0-20/60-80")
ang_i = np.interp(xd, [a["x"] for a in ang], [a["value"] for a in ang]) if ang else np.ones_like(xd)
blk = R["raa"]["mbd"]["sph_jet"]; Es = np.array(R["edges"]["sph_jet"]); xs = 0.5 * (Es[1:] + Es[:-1])
rv, rm = np.array(blk["oo_vac"]["raa"]), np.array(blk["oo_med"]["raa"])
ev_, em_ = np.array(blk["oo_vac"]["err"]), np.array(blk["oo_med"]["err"])
ours_bias = rv[0] / rv[3]
ours_bias_d = np.interp(xd, xs, ours_bias)
bb = R["raa"]["b"]["sph_jet"]; qb = np.array(bb["oo_med"]["raa"])[0] / np.array(bb["oo_vac"]["raa"])[0]
eqb = qb * np.hypot(np.array(bb["oo_med"]["err"])[0] / np.array(bb["oo_med"]["raa"])[0],
                    np.array(bb["oo_vac"]["err"])[0] / np.array(bb["oo_vac"]["raa"])[0])
qr = (rm[0] / rm[3]) / (rv[0] / rv[3])
eqr = qr * np.sqrt(sum((e[c] / r[c]) ** 2 for r, e in ((rm, em_), (rv, ev_)) for c in (0, 3)))
n_d, n_s = nloc(pS, xd), nloc(pS, xs)
eps = lambda q, n: 1 - np.clip(q, 1e-6, None) ** (1 / (n - 1))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
xx = np.linspace(15, 70, 200)
for e_, c in ((0.01, "C0"), (0.02, "C1"), (0.04, "C2"), (0.06, "C3")):
    ax[0].plot(xx, (1 - e_) ** (nloc(pS, xx) - 1), color=c, label=f"ε = {e_:.0%} of p_T")
    ax[0].plot(xx, (1 - 1.5 * e_ * 30 / xx) ** (nloc(pS, xx) - 1), color=c, ls=":", lw=0.9)
ax[0].errorbar(xd, vd / ang_i, ed / ang_i, fmt="*", ms=9, color="k", label="sPHENIX R_CP / Angantyr bias")
ax[0].errorbar(xd + 0.6, vd / ours_bias_d, ed / ours_bias_d, fmt="*", ms=9, color="0.55", label="sPHENIX R_CP / our bias")
m_ = xs > 19
ax[0].errorbar(xs[m_], qb[m_], eqb[m_], fmt="o", color="C3", label="model Q, 0-20 % (b)")
ax[0].set_ylim(0.2, 1.15); ax[0].set_xlabel("jet $p_T$ [GeV]"); ax[0].set_ylabel("suppression")
ax[0].set_title("fixed fractional loss with sPHENIX's n(p_T); dotted: fixed ΔE (ε at 30 GeV ×1.5)", fontsize=9)
ax[0].legend(fontsize=7)
ax[1].errorbar(xd, 100 * eps(vd / ang_i, n_d), 100 * ed / ang_i / (n_d - 1), fmt="*", ms=9, color="k", label="data / Angantyr bias")
ax[1].errorbar(xd + 0.6, 100 * eps(vd / ours_bias_d, n_d), 100 * ed / ours_bias_d / (n_d - 1), fmt="*", ms=9, color="0.55", label="data / our bias")
ax[1].errorbar(xs[m_], 100 * eps(qb, n_s)[m_], 100 * (eqb / (n_s - 1))[m_], fmt="o", color="C3", label="model, 0-20 % (b)")
ax[1].errorbar(xs[m_] + 0.6, 100 * eps(qr, n_s)[m_], 100 * (eqr / (n_s - 1))[m_], fmt="s", ms=4, color="C1", label="model, R_CP double ratio")
ax[1].axhline(0, color="k", lw=0.8); ax[1].set_xlabel("jet $p_T$ [GeV]"); ax[1].set_ylabel("effective ε [% of p_T]")
ax[1].set_ylim(-3, 10); ax[1].legend(fontsize=7); ax[1].set_title("quenching as an effective fractional p_T loss", fontsize=9)
fig.tight_layout(); plt.show()

print("bin         n(sPHENIX fit)   data R_CP/Angantyr -> eps    data R_CP/our bias -> eps")
for i in range(len(xd)):
    print(f"  {rows[i]['lo']:.0f}-{rows[i]['hi']:.0f}        {n_d[i]:5.1f}          {vd[i]/ang_i[i]:.2f} -> {100*eps(vd[i]/ang_i[i], n_d[i]):4.1f} %"
          f"            {vd[i]/ours_bias_d[i]:.2f} -> {100*eps(vd[i]/ours_bias_d[i], n_d[i]):4.1f} %")
print("model bin   n            Q 0-20 % (b) -> eps          Q R_CP -> eps")
for i in np.where(m_)[0]:
    print(f"  {Es[i]:.0f}-{Es[i+1]:.0f}        {n_s[i]:5.1f}       {qb[i]:.2f}±{eqb[i]:.2f} -> {100*eps(qb[i], n_s[i]):4.1f} %"
          f"          {qr[i]:.2f}±{eqr[i]:.2f} -> {100*eps(qr[i], n_s[i]):4.1f} %")

## Why is the fractional loss nearly flat? Two checks in the model

**Jet flavour.** Each full jet is tagged by the nearest outgoing parton of the hard scattering
(`initiators/`, pstat 1000) within ΔR < 0.4: quark (or antiquark) or gluon. About 1.5 % of jets have
no hard parton that close and are left out of both. At high x_T the valence quarks take over, and
quarks lose about C_F/C_A = 4/9 as much as gluons. So a flat loss per flavour gives a total ε that
falls with p_T.

**Cone size.** STAR-like charged jets with R = 0.2 and R = 0.5 (|η| < 1 − R; holes subtracted with
the 2/3 charged fraction). Energy carried to the cone edge and the medium response flowing back in
depend on R.

Quenching here is always Q = `oo_med`/`oo_vac` in 0–20 % by impact parameter (no selection bias), and
ε = 1 − Q^(1/(n−1)) with n from the matching p+p spectrum of `pp_ref` (per flavour, per R).

In [ ]:
Rj = json.loads(Path(RES).read_text())
def stitched_pp(key):
    h = ev[key]; S_ = (ev["w"][:, None] * h).sum(0); S2_ = ((ev["w"][:, None] * h) ** 2).sum(0)
    return S_, np.sqrt(S2_)
def q_b(key, c=0):
    bb = Rj["raa"]["b"][key]
    m, em = np.array(bb["oo_med"]["raa"])[c], np.array(bb["oo_med"]["err"])[c]
    v, evv = np.array(bb["oo_vac"]["raa"])[c], np.array(bb["oo_vac"]["err"])[c]
    q = m / v; return q, q * np.hypot(em / m, evv / v)

Es = np.array(Rj["edges"]["sph_jet"]); xs = 0.5 * (Es[1:] + Es[:-1]); sel = Es[:-1] >= 19
tot, _ = stitched_pp("sph_jet"); qq, eqq = stitched_pp("sph_jet_q"); gg, egg = stitched_pp("sph_jet_g")
for name in ("oo_vac", "oo_med"):
    e_ = dict(np.load(f"../results/oo_analysis_rhic_{name}.npz")); W = e_["w"] * e_["ncoll"]
    t_ = (W[:, None] * e_["sph_jet"]).sum(0); g_ = (W[:, None] * e_["sph_jet_g"]).sum(0)
    print(f"{name:7s} gluon share of full jets: " + " ".join(f"{a:.2f}" for a in g_ / t_))
print(f"pp_ref  gluon share of full jets: " + " ".join(f"{a:.2f}" for a in gg / tot))
print("bins:", " ".join(f"{a:g}-{b:g}" for a, b in zip(Es[:-1], Es[1:])))
conv_ = 1e9 / np.diff(Es) / 1.4
pq = fit_simple(Es[:-1], Es[1:], qq * conv_, eqq * conv_); pg = fit_simple(Es[:-1], Es[1:], gg * conv_, egg * conv_)
Qq, eQq = q_b("sph_jet_q"); Qg, eQg = q_b("sph_jet_g"); Qt, eQt = q_b("sph_jet")
nq, ng, nt = nloc(pq, xs), nloc(pg, xs), nloc(FITS[km][0], xs)
eps = lambda q, n: 1 - np.clip(q, 1e-6, None) ** (1 / (n - 1))
deps = lambda q, e, n: e / q / (n - 1)

fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
ax[0].plot(xs, gg / tot, "o-", color="C2", label="gluon share, pp_ref")
ax[0].plot(xs, qq / tot, "s-", color="C0", label="quark share, pp_ref")
ax[0].set_ylim(0, 1); ax[0].set_xlabel("jet $p_T$ [GeV]"); ax[0].legend(fontsize=8)
ax[0].set_title("flavour of R = 0.4 full jets (nearest hard parton)", fontsize=9)
ax[1].plot(xs, nq, color="C0", label="n, quark jets"); ax[1].plot(xs, ng, color="C2", label="n, gluon jets")
ax[1].plot(xs, nt, color="k", ls=":", label="n, all"); ax[1].set_xlabel("jet $p_T$ [GeV]"); ax[1].set_ylabel("local power n")
ax[1].legend(fontsize=8); ax[1].set_title("pp_ref spectra per flavour (fits)", fontsize=9)
for (Q, eQ, n_, lab, c, dx) in ((Qq, eQq, nq, "quark jets", "C0", -0.5), (Qg, eQg, ng, "gluon jets", "C2", 0.5), (Qt, eQt, nt, "all", "k", 0)):
    ax[2].errorbar(xs[sel] + dx, 100 * eps(Q, n_)[sel], 100 * deps(Q, eQ, n_)[sel], fmt="o", color=c, label=lab)
ax[2].axhline(0, color="k", lw=0.8); ax[2].set_xlabel("jet $p_T$ [GeV]"); ax[2].set_ylabel("effective ε [% of p_T]")
ax[2].legend(fontsize=8); ax[2].set_title("model ε per flavour, 0-20 % by b", fontsize=9)
fig.tight_layout(); plt.show()

print("bin        Q quark        Q gluon        eps quark   eps gluon   eps all")
for i in np.where(sel)[0]:
    print(f"  {Es[i]:.0f}-{Es[i+1]:.0f}    {Qq[i]:.2f}±{eQq[i]:.2f}     {Qg[i]:.2f}±{eQg[i]:.2f}      {100*eps(Qq[i], nq[i]):4.1f}%      "
          f"{100*eps(Qg[i], ng[i]):4.1f}%     {100*eps(Qt[i], nt[i]):4.1f}%")
w = 1 / deps(Qq, eQq, nq)[sel] ** 2; mq = np.sum(eps(Qq, nq)[sel] * w) / w.sum(); smq = w.sum() ** -0.5
w = 1 / deps(Qg, eQg, ng)[sel] ** 2; mg = np.sum(eps(Qg, ng)[sel] * w) / w.sum(); smg = w.sum() ** -0.5
print(f"mean over 19-60 GeV: eps quark {100*mq:.2f} ± {100*smq:.2f} %, eps gluon {100*mg:.2f} ± {100*smg:.2f} %"
      f"  (C_A/C_F would give gluon/quark = 2.25)")
# the composition alone: eps_q fixed, eps_g = 2.25 eps_q
fg = gg / tot; comp = (1 - fg) + 2.25 * fg
i20, i47 = np.searchsorted(Es, 19) , np.searchsorted(Es, 43)
print(f"composition alone (eps_g = 2.25 eps_q): eps_all(43-51) / eps_all(19-24) = {comp[i47] / comp[i20]:.2f}")

In [ ]:
Ec = np.array(Rj["edges"]["ch_jet"]); xc_ = 0.5 * (Ec[1:] + Ec[:-1]); selc = Ec[:-1] >= 8
fig, ax = plt.subplots(1, 2, figsize=(12, 4.6))
out = {}
for key, c, dx in (("ch_jet_R0.2", "C1", -0.4), ("ch_jet_R0.5", "C4", 0.4)):
    S_, eS_ = stitched_pp(key); R_ = float(key.split("R")[1]); conv_c = 1e9 / np.diff(Ec) / (2 * (1 - R_))
    pc = fit_simple(Ec[:-1], Ec[1:], S_ * conv_c, eS_ * conv_c); n_c = nloc(pc, xc_)
    Q, eQ = q_b(key)
    out[key] = (Q, eQ, n_c)
    ax[0].plot(xc_, n_c, color=c, label=f"n, charged jets {key[-4:]}")
    ax[1].errorbar(xc_[selc] + dx, 100 * eps(Q, n_c)[selc], 100 * deps(Q, eQ, n_c)[selc], fmt="o", color=c, label=f"charged jets {key[-4:]}")
ax[0].set_xlabel("charged-jet $p_T$ [GeV]"); ax[0].set_ylabel("local power n"); ax[0].legend(fontsize=8)
ax[0].set_title("pp_ref charged-jet spectra (fits)", fontsize=9)
ax[1].axhline(0, color="k", lw=0.8); ax[1].set_xlabel("charged-jet $p_T$ [GeV]"); ax[1].set_ylabel("effective ε [% of p_T]")
ax[1].legend(fontsize=8); ax[1].set_title("model ε vs cone size, 0-20 % by b (holes: 2/3 charged)", fontsize=9)
fig.tight_layout(); plt.show()
print("bin       Q R=0.2        Q R=0.5        eps R=0.2   eps R=0.5")
for i in np.where(selc)[0]:
    a, b = out["ch_jet_R0.2"], out["ch_jet_R0.5"]
    print(f"  {Ec[i]:.0f}-{Ec[i+1]:.0f}    {a[0][i]:.2f}±{a[1][i]:.2f}     {b[0][i]:.2f}±{b[1][i]:.2f}      "
          f"{100*eps(a[0][i], a[2][i]):5.1f}%      {100*eps(b[0][i], b[2][i]):5.1f}%")
ic = Rj["icp"]["oo_med"]["icp"]; iv = Rj["icp"]["oo_vac"]["icp"]
print(f"STAR-like h+jet I_CP 0-10/40-60 (EPD), 8-19 GeV: R=0.2 med {ic['hjet_R0.2_8_19'][0][0]:.2f}±{ic['hjet_R0.2_8_19'][0][1]:.2f}"
      f" (vac {iv['hjet_R0.2_8_19'][0][0]:.2f}), R=0.5 med {ic['hjet_R0.5_8_19'][0][0]:.2f}±{ic['hjet_R0.5_8_19'][0][1]:.2f}"
      f" (vac {iv['hjet_R0.5_8_19'][0][0]:.2f}); STAR R=0.5: 0.806, R=0.2: see HEPData Fig. 3")

## Reading (2026-11-25, all jobs: phase 3 + 3b + 3e; model results not vetted, sPHENIX preliminary)

**1. The steepening is large, and every spectrum has it.** The local power n(p_T) roughly doubles
between 20 and 50 GeV:

| p_T (GeV) | 15 | 20 | 30 | 40 | 50 | 60 |
|---|---|---|---|---|---|---|
| sPHENIX (R = 0.4, \|η\| < 0.7) | 5.8 | 6.6 | 8.5 | 11.0 | 14.5 | 19.8 |
| STAR (R = 0.5, \|η\| < 0.5) | 6.4 | 6.9 | 8.3 | 10.2 | 12.8 | – |
| STAR (R = 0.5, 0.5 < \|η\| < 0.9) | 6.5 | 7.3 | 9.2 | 11.7 | 15.2 | – |
| our `pp_ref` (R = 0.4, \|η\| < 0.7) | 6.2 | 7.0 | 8.9 | 11.5 | 15.1 | 20.5 |

- All fits have m ≈ 8–11 for the (1 − x_T)^m factor. The point-by-point n from neighbouring bins
  follow the fits.
- At the LHC (5 TeV, x_T < 0.03 at these p_T) n stays at about 5–6. At RHIC, x_T reaches 0.5–0.6, where
  the parton densities fall steeply.
- **Our p+p reference has the same n(p_T) as the data.**
- **In absolute terms,** compared in each experiment's own jet definition and bins (`pp_jet_spectra.py`):
  - against STAR, R = 0.5, |η| < 0.5: 0.86–1.14 up to 31 GeV, then 0.88 at 31–37 GeV and 0.70–0.71
    at 37–52 GeV. That is the high-p_T shortfall the hard tune found with the vacuum chain (0.65–0.8
    at 35–48 GeV), now in the full chain.
  - against STAR, 0.5 < |η| < 0.9: 0.86–1.22, without a clear trend.
  - against sPHENIX, R = 0.4, |η| < 0.7: 1.53 at 15–19 GeV, falling to 1.00–1.04 (± 0.02) at 34–63 GeV and
    0.78 at 63–72 GeV.
- **STAR against sPHENIX directly,** with the model removing the different jet definitions (section
  "STAR against sPHENIX"):
  - STAR at |η| < 0.5 is above sPHENIX by about ×1.3 at all p_T: 1.23–1.37 at 16–37 GeV, 1.4–1.5
    at 37–52 GeV with ±0.3–0.4.
  - STAR at 0.5 < |η| < 0.9 is above by about ×1.2: 1.08–1.30, and 0.96 ± 0.5 in the last bin.
  - So the experiments differ mainly in **normalisation, by 20–30 %, not in shape.** That is within
    sPHENIX's systematics (MBD cross section and JES; the shaded band) and well outside STAR's 5.6 %
    normalisation uncertainty.
- **The model against both:**
  - it lies between the two at 15–30 GeV;
  - it agrees with sPHENIX at 34–63 GeV;
  - it is about 30 % below STAR's |η| < 0.5 jets at 37–52 GeV, where it also falls a little faster
    than STAR. That last point is the one real shape difference, and the hard tune found it too.
- **So n(p_T), which these systematics hardly move, is the robust comparison.** There data and model agree.

**2. What that does to the suppression.** A constant fractional loss ε gives Q ≈ (1 − ε)^(n−1):
- ε = 4 % gives 0.82 at 20 GeV, 0.62 at 40 GeV and 0.33 at 70 GeV;
- ε = 1 % gives 0.94 → 0.86 over 20–60 GeV.

So at RHIC a falling R_AA or R_CP with p_T does not mean that higher-p_T jets lose more. A constant
or even slowly falling fractional loss gives it. That is the opposite of the LHC pattern, where n is
flat and a falling ε makes R_AA rise.

**3. Read as a fractional loss:**
- **The data** (sPHENIX R_CP divided by the selection bias): ε ≈ 5–7 % at 19–30 GeV and ≈ 4–5 % at
  36–60 GeV with Angantyr's bias; with ours ≈ 5–6 % at 19–30 GeV and 2–5 % at 30–60 GeV (our bias
  is larger at high p_T, about 0.8 against 0.9 below 36 GeV). Roughly flat or slowly falling. The steep drop to
  0.35 at 51–60 GeV needs no extra loss: with n ≈ 17 it is ε ≈ 5 %, like the other bins.
- **The model:** ε ≈ 0.4–2 % in 0–20 % by impact parameter, flat within errors: 0.8–1.1 % at
  36–60 GeV (± ≈ 0.4 %). With 3e the earlier 43–51 GeV dip (Q 0.81, ε 1.7 %) is gone (Q 0.87 ± 0.05,
  ε 1.1 %): it was a fluctuation. The R_CP double ratio gives 0–1.8 %.
- **Model against data:** our Matter + LBT takes about 2–5× less fractional p_T out of the
  R = 0.4 cone than the data imply, at every p_T. That is the quantitative form of the model's
  missing quenching.

**4. Flavour (model, all jobs, 0–20 % by impact parameter).**
- **Composition:** quark jets dominate more and more. The gluon share of `pp_ref` R = 0.4 jets falls
  from 34 % at 10–12 GeV to 24 % at 19–24 GeV, 15 % at 30–36 GeV and 5 % at 51–60 GeV. This is the
  model's own composition (the hard tune's PDF set, `PDF:pSet` 20), not checked against a standalone
  Pythia. Gluon jets have the steeper spectrum (n ≈ 8 against 7 at 20 GeV).
- **What composition alone would do:** with gluons losing C_A/C_F = 2.25 times the quarks' fraction,
  this composition makes the total ε fall by about 15 % from 19–24 to 43–51 GeV.
  - That is about half of the fall seen with our bias (5.1 → 3.7 %).
  - It is about a third of the fall seen with Angantyr's (7.0 → 3.8 %).
- **The model shows at most a weak Casimir ordering:** ε_quark = 0.92 ± 0.24 % and ε_gluon =
  1.24 ± 0.47 %, averaged over 19–60 GeV: gluon/quark ≈ 1.3 ± 0.6. Gluon jets losing 2.25× more
  would mean about 2.1 %, 1.8σ from the measured gluon value, so neither 1 nor 2.25 is excluded.
  (With phase 3 alone it was 1.1 ± 0.4 against 0.7 ± 0.5 %.) The gluon jets' Q is noisy
  (±0.09–0.18; 1.08 at 19–24 GeV, 0.58 ± 0.11 at 43–51 GeV, where gluons are 10 % of the jets).
- **Possible reasons the gluons don't lose more:** gluon jets are wider and may get more of the medium
  response back into the cone; LBT and MATTER's colour factors in a short medium; or statistics. It
  is worth a direct check (the energy of recoils and holes in the cone per flavour).

**5. Cone size (model, STAR-like charged jets R = 0.2 and R = 0.5, 0–20 % by impact parameter).**
- No difference within the errors (Q ± 0.09–0.12 per bin).
- Both cone sizes show Q > 1 at 8–10 GeV (1.10 ± 0.10 for both). There the medium response, or the 2/3
  hole approximation, adds jets.
- The STAR-like h+jet I_CP is about the same for R = 0.2 and 0.5 with quenching (0.84 ± 0.11 and
  0.87 ± 0.10; vacuum 0.88 and 0.99), both within 1σ of STAR (0.79, 0.81). At this statistics the R
  comparison doesn't resolve anything.

**What this says about the question.** A fractional loss that falls weakly with p_T is what both the
data and the expectation give.
- **The data** allow ΔE ∝ p_T^(0.3–0.8), and the change in jet flavour alone accounts for 15 % of the
  fall over 20–50 GeV.
- **The model** shows no significant Casimir ordering between quark and gluon jets (1.3 ± 0.6) and
  no R dependence it can resolve. With its small ε (≈ 1 %) it lacks the precision to test either,
  even with 3b and 3e; the in-cone energy balance of recoils and holes per flavour is the direct test.

**Caveats**
- **A fixed shift is a simplification.** With fluctuating energy loss, a steep spectrum favours jets
  that lost little, so the ε here is an effective, smaller-than-mean loss. Data and model are
  converted the same way.
- **R_CP is relative** to the 60–80 % class, which is itself slightly quenched.
- **The data are preliminary,** read from figures. sPHENIX's statistical bars are drawing sizes, and
  its systematics are correlated between bins, so they barely change n.